# Optimasi Rute Puskesmas (VRP) - Memetic Algorithm
**Klaster:** Barat | **Constraint:** 10 Jam Kerja, 20 Menit Pelayanan, & Prioritas Induk Inap

In [30]:
import pandas as pd
import numpy as np
import json
import random
import os

# ==========================================
# 1. KONFIGURASI PATH & PARAMETER VRP
# ==========================================
KLASTER = 'Utara'  # 👉 Tinggal ganti ke 'Pusat', 'Selatan', 'Timur', 'Utara' secara bergantian

# Relative path mundur dari folder 'algorithms'
file_koordinat = '../data/koordinat_eas.csv'
file_matriks_jarak = f'../data/matriks_jarak_riil_{KLASTER.lower()}.csv'
file_matriks_waktu = f'../data/datamatriks_waktu_{KLASTER.lower()}.csv'

# Path folder output (sejajar dengan folder data)
folder_output = '../output'
file_output_json = f'{folder_output}/rute_ma.json'  # 👉 Semua masuk ke 1 file ini

# Auto-create folder output kalau belum ada
os.makedirs(folder_output, exist_ok=True)

MAX_WORKING_TIME = 600     # 10 jam = 600 menit
SERVICE_TIME = 20          # 20 menit per puskesmas
PENALTY_VIOLATION = 10000  # Penalti besar untuk rute melanggar prioritas

# Parameter Genetic Algorithm
POP_SIZE = 50
GENERATIONS = 100
CROSSOVER_RATE = 0.85
MUTATION_RATE = 0.15
TOURNAMENT_SIZE = 3

print(f"✅ Setup parameter untuk klaster {KLASTER.upper()} berhasil di-load!")

✅ Setup parameter untuk klaster UTARA berhasil di-load!


In [31]:
# ==========================================
# 2. LOAD DATA & PENENTUAN PRIORITAS
# ==========================================
df_koordinat = pd.read_csv(file_koordinat)
matriks_jarak = pd.read_csv(file_matriks_jarak, index_col=0)
matriks_waktu = pd.read_csv(file_matriks_waktu, index_col=0)

# [PENTING] Cleansing Koordinat Ekstra Kuat
def bersihkan_latitude(val):
    if pd.isna(val): return 0.0
    val_bersih = str(val).replace(',', '').replace('.', '').replace(' ', '')
    angka_saja = val_bersih.replace('-', '')
    
    if len(angka_saja) > 1:
        hasil = angka_saja[:1] + '.' + angka_saja[1:]
    else:
        hasil = angka_saja
        
    return float('-' + hasil) if '-' in str(val) else float(hasil)

def bersihkan_longitude(val):
    if pd.isna(val): return 0.0
    val_bersih = str(val).replace(',', '').replace('.', '').replace(' ', '')
    
    if len(val_bersih) > 3:
        hasil = val_bersih[:3] + '.' + val_bersih[3:]
    else:
        hasil = val_bersih
        
    return float(hasil)

# Dictionary untuk mempermudah pencarian koordinat [lat, lon]
dict_koordinat = {}
for _, row in df_koordinat.iterrows():
    lat = bersihkan_latitude(row['Latitude'])
    lon = bersihkan_longitude(row['Longitude'])
    dict_koordinat[row['Nama Puskesmas'].strip()] = [lat, lon]

# Fungsi penentu tier prioritas
def get_priority_tier(nama_lokasi):
    nama_lower = nama_lokasi.lower()
    if 'uptd' in nama_lower or 'gudang farmasi' in nama_lower:
        return 0  # Depot
    elif 'inap' in nama_lower:
        return 1  # Prioritas 1: Induk Inap
    elif 'pustu' in nama_lower:
        return 3  # Prioritas 3: Pustu
    else:
        return 2  # Prioritas 2: Induk Jalan

# Setup list lokasi
semua_lokasi = list(matriks_waktu.index)
depot = semua_lokasi[0]
daftar_puskesmas = semua_lokasi[1:]

print(f"✅ Data {KLASTER} siap! Jumlah Puskesmas: {len(daftar_puskesmas)}")

✅ Data Utara siap! Jumlah Puskesmas: 22


In [32]:
# ==========================================
# 3. FUNGSI LOGIKA VRP & CONSTRAINT CHECK
# ==========================================
def check_priority_violation(route_nodes):
    for i in range(len(route_nodes)):
        for j in range(i + 1, len(route_nodes)):
            if get_priority_tier(route_nodes[i]) > get_priority_tier(route_nodes[j]):
                return True
    return False

def split_giant_tour(chromosome):
    routes = []
    current_route = []
    current_time = 0
    current_dist = 0
    prev_node = depot
    
    for node in chromosome:
        time_to_node = matriks_waktu.loc[prev_node, node]
        time_to_depot = matriks_waktu.loc[node, depot]
        potential_time = current_time + time_to_node + SERVICE_TIME + time_to_depot
        
        if potential_time <= MAX_WORKING_TIME:
            current_route.append(node)
            current_time += time_to_node + SERVICE_TIME
            current_dist += matriks_jarak.loc[prev_node, node]
            prev_node = node
        else:
            if current_route:
                current_time += matriks_waktu.loc[prev_node, depot]
                current_dist += matriks_jarak.loc[prev_node, depot]
                routes.append({'route': [depot] + current_route + [depot], 'time': current_time, 'distance': current_dist, 'priority_violation': check_priority_violation(current_route)})
            
            # Start kurir baru
            current_route = [node]
            current_time = matriks_waktu.loc[depot, node] + SERVICE_TIME
            current_dist = matriks_jarak.loc[depot, node]
            prev_node = node
            
    if current_route:
        current_time += matriks_waktu.loc[prev_node, depot]
        current_dist += matriks_jarak.loc[prev_node, depot]
        routes.append({'route': [depot] + current_route + [depot], 'time': current_time, 'distance': current_dist, 'priority_violation': check_priority_violation(current_route)})
        
    return routes

def evaluate_fitness(chromosome):
    routes = split_giant_tour(chromosome)
    total_time = sum(r['time'] for r in routes)
    penalty = sum(PENALTY_VIOLATION for r in routes if r['priority_violation'])
    return total_time + penalty

In [33]:
# ==========================================
# 4. LOCAL SEARCH (2-OPT)
# ==========================================
def local_search_2opt(route_dict):
    full_route = route_dict['route']
    nodes = full_route[1:-1]
    
    if len(nodes) < 2: return route_dict
        
    best_time = route_dict['time']
    best_nodes = nodes.copy()
    improved = True
    
    while improved:
        improved = False
        for i in range(len(best_nodes)):
            for j in range(i + 1, len(best_nodes)):
                new_nodes = best_nodes.copy()
                new_nodes[i:j+1] = list(reversed(new_nodes[i:j+1]))
                
                if check_priority_violation(new_nodes): continue
                    
                new_time = 0
                prev = depot
                for n in new_nodes:
                    new_time += matriks_waktu.loc[prev, n] + SERVICE_TIME
                    prev = n
                new_time += matriks_waktu.loc[prev, depot]
                
                if new_time < best_time and new_time <= MAX_WORKING_TIME:
                    best_time = new_time
                    best_nodes = new_nodes
                    improved = True
                    
    prev = depot
    final_dist = 0
    for n in best_nodes:
        final_dist += matriks_jarak.loc[prev, n]
        prev = n
    final_dist += matriks_jarak.loc[prev, depot]
    
    return {'route': [depot] + best_nodes + [depot], 'time': best_time, 'distance': final_dist, 'priority_violation': False}

In [34]:
# ==========================================
# 5. KOMPONEN GENETIC ALGORITHM
# ==========================================
def create_individual():
    tier1 = [n for n in daftar_puskesmas if get_priority_tier(n) == 1]
    tier2 = [n for n in daftar_puskesmas if get_priority_tier(n) == 2]
    tier3 = [n for n in daftar_puskesmas if get_priority_tier(n) == 3]
    random.shuffle(tier1)
    random.shuffle(tier2)
    random.shuffle(tier3)
    return tier1 + tier2 + tier3

def tournament_selection(population):
    selected = random.sample(population, TOURNAMENT_SIZE)
    return min(selected, key=evaluate_fitness)

def ordered_crossover(p1, p2):
    size = len(p1)
    start, end = sorted(random.sample(range(size), 2))
    child = [None] * size
    child[start:end+1] = p1[start:end+1]
    
    p2_pointer = 0
    for i in range(size):
        if child[i] is None:
            while p2[p2_pointer] in child:
                p2_pointer += 1
            child[i] = p2[p2_pointer]
    return child

def swap_mutation(chromosome):
    if random.random() < MUTATION_RATE:
        idx1, idx2 = random.sample(range(len(chromosome)), 2)
        chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
    return chromosome

In [35]:
# ==========================================
# 6. MAIN ENGINE: EKSEKUSI MEMETIC
# ==========================================
import time # Import module waktu untuk hitung komputasi

print(f"🚀 Menjalankan Memetic Algorithm ({GENERATIONS} Generasi)...")
start_time = time.time() # ⏱️ Mulai timer!

population = [create_individual() for _ in range(POP_SIZE)]
riwayat_konvergensi = [] # Tempat menyimpan sejarah grafik

for gen in range(GENERATIONS):
    new_population = []
    population.sort(key=evaluate_fitness)
    new_population.append(population[0]) # Elitism
    
    while len(new_population) < POP_SIZE:
        p1, p2 = tournament_selection(population), tournament_selection(population)
        child = ordered_crossover(p1, p2) if random.random() < CROSSOVER_RATE else p1.copy()
        child = swap_mutation(child)
        
        # Local Search 2-Opt (Pendidikan anak)
        sub_routes = split_giant_tour(child)
        optimized_tour = []
        for r in sub_routes:
            opt_r = local_search_2opt(r)
            optimized_tour.extend(opt_r['route'][1:-1])
            
        new_population.append(optimized_tour)
        
    population = new_population
    
    # Catat nilai terbaik di generasi ini untuk grafik konvergensi
    best_current = min(population, key=evaluate_fitness)
    best_fit = evaluate_fitness(best_current)
    riwayat_konvergensi.append(round(best_fit, 2))
    
    if (gen + 1) % 10 == 0:
        print(f"🔄 Generasi {gen+1:03d} | Waktu Terpendek Sementara: {best_fit:.2f} menit")

end_time = time.time() # ⏱️ Stop timer!
waktu_komputasi = round(end_time - start_time, 3) # Hitung selisih dalam detik

print("✅ Evolusi Selesai!")
print(f"⏱️ Waktu Komputasi: {waktu_komputasi} detik")

🚀 Menjalankan Memetic Algorithm (100 Generasi)...
🔄 Generasi 010 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 020 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 030 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 040 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 050 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 060 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 070 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 080 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 090 | Waktu Terpendek Sementara: 577.53 menit
🔄 Generasi 100 | Waktu Terpendek Sementara: 577.53 menit
✅ Evolusi Selesai!
⏱️ Waktu Komputasi: 174.228 detik


In [36]:
# ==========================================
# 7. PARSING HASIL AKHIR & MERGE KE 1 FILE JSON
# ==========================================
best_chromosome = min(population, key=evaluate_fitness)
final_sub_routes = split_giant_tour(best_chromosome)

total_waktu, total_jarak = 0, 0
rute_json = []

for idx, r in enumerate(final_sub_routes):
    r_opt = local_search_2opt(r)
    total_waktu += r_opt['time']
    total_jarak += r_opt['distance']
    
    koordinat_list = [dict_koordinat[nama.strip()] for nama in r_opt['route']]
    rute_json.append({
        "id_kurir": idx + 1,
        "waktu_tempuh_menit": round(r_opt['time'], 2),
        "jarak_tempuh_km": round(r_opt['distance'], 2),
        "urutan_kunjungan": r_opt['route'],
        "koordinat_kunjungan": koordinat_list
    })

# Format data khusus untuk klaster ini sesuai template barumu
data_klaster_ini = {
    "total_kurir": len(final_sub_routes),
    "waktu_komputasi_detik": waktu_komputasi,
    "total_waktu_semua_menit": round(total_waktu, 2),
    "total_jarak_semua_km": round(total_jarak, 2),
    "riwayat_konvergensi": riwayat_konvergensi,
    "rute_per_kurir": rute_json
}

# --- PROSES MERGE DATA ---
if os.path.exists(file_output_json):
    with open(file_output_json, 'r') as f:
        try:
            database_rute = json.load(f)
            # Jaga-jaga kalau file lama formatnya pakai key 'hasil_klaster' bukan 'hasil_per_klaster'
            if "hasil_per_klaster" not in database_rute:
                database_rute["hasil_per_klaster"] = {}
        except json.JSONDecodeError:
            database_rute = {"algoritma": "Memetic Algorithm (MA)", "hasil_per_klaster": {}}
else:
    database_rute = {"algoritma": "Memetic Algorithm (MA)", "hasil_per_klaster": {}}

# Masukkan data ke klaster yang dituju
database_rute["hasil_per_klaster"][KLASTER] = data_klaster_ini

# Simpan!
with open(file_output_json, 'w') as f:
    json.dump(database_rute, f, indent=4)

print("="*40)
print(f"🎉 JSON BERHASIL DISIMPAN & DI-MERGE!")
print(f"📁 Path File         : {file_output_json}")
print(f"📍 Klaster Tersimpan : {KLASTER}")
print("="*40)

🎉 JSON BERHASIL DISIMPAN & DI-MERGE!
📁 Path File         : ../output/rute_ma.json
📍 Klaster Tersimpan : Utara
